In [1]:
import os
import pandas as pd
from pathlib import Path
import shutil

In [22]:
os.chdir('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv')
os.listdir()

['calc_case_description_test_set.csv',
 'calc_case_description_train_set.csv',
 'dicom_info.csv',
 'mass_case_description_test_set.csv',
 'mass_case_description_train_set.csv',
 'meta.csv']

In [23]:
train_data = pd.read_csv('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv\\mass_case_description_train_set.csv')
test_data = pd.read_csv('C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\csv\\mass_case_description_test_set.csv')

In [24]:
train_data.columns

Index(['patient_id', 'breast_density', 'left or right breast', 'image view',
       'abnormality id', 'abnormality type', 'mass shape', 'mass margins',
       'assessment', 'pathology', 'subtlety', 'image file path',
       'cropped image file path', 'ROI mask file path'],
      dtype='object')

### Extract Series UID

In [25]:

def extract_uid(path):
    if pd.isna(path):
        return None
    return str(path).split('/')[-2]

# Extract UID
train_uid = train_data['image file path'].apply(extract_uid)
test_uid = test_data['image file path'].apply(extract_uid)

# Insert as first column
train_data.insert(0, 'SeriesUID', train_uid)
test_data.insert(0, 'SeriesUID', test_uid)

# Check


In [26]:
print(train_data[['SeriesUID']].head())
print(test_data[['SeriesUID']].head())

                                           SeriesUID
0  1.3.6.1.4.1.9590.100.1.2.342386194811267636608...
1  1.3.6.1.4.1.9590.100.1.2.359308329312397897125...
2  1.3.6.1.4.1.9590.100.1.2.891800462110225318343...
3  1.3.6.1.4.1.9590.100.1.2.295360926313492745441...
4  1.3.6.1.4.1.9590.100.1.2.410524754913057908920...
                                           SeriesUID
0  1.3.6.1.4.1.9590.100.1.2.245063149211255120613...
1  1.3.6.1.4.1.9590.100.1.2.859522146111705060178...
2  1.3.6.1.4.1.9590.100.1.2.221311896128932948279...
3  1.3.6.1.4.1.9590.100.1.2.239949064412092068706...
4  1.3.6.1.4.1.9590.100.1.2.215081818713600536113...


### Merge Train and Test data first

As, It is not Tabular data, no scaling or imputation has been done so, there is no risk of data Leakage. 

In [27]:
all_data = pd.concat([train_data, test_data], ignore_index=True)

In [28]:
all_data['pathology'].value_counts()

pathology
MALIGNANT                  784
BENIGN                     771
BENIGN_WITHOUT_CALLBACK    141
Name: count, dtype: int64

In [29]:
all_data_clean = all_data[all_data['pathology'] != 'BENIGN_WITHOUT_CALLBACK' ].copy()

In [30]:
all_data_clean.shape

(1555, 15)

In [31]:
jpeg_root = Path(r"C:\Users\moham\Downloads\archive\jpeg")

uids = set(all_data_clean['SeriesUID'].astype(str))

jpeg_folders = {folder.name for folder in jpeg_root.iterdir() if folder.is_dir()}

matched_uids = uids.intersection(jpeg_folders)

print("UIDs in dataframe:", len(uids))
print("JPEG folders:", len(jpeg_folders))
print("Matches:", len(matched_uids))
print("Missing:", len(uids) - len(matched_uids))

UIDs in dataframe: 1473
JPEG folders: 6774
Matches: 1473
Missing: 0


In [33]:
jpeg_root = Path(r"C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\jpeg")
target_root = Path(r"C:\\Users\\moham\\Downloads\\Projects\\Breast_Cancer_Prediction\\Dataset\\archive\\jpeg_selected")

target_root.mkdir(exist_ok=True)

for uid in matched_uids:
    src = jpeg_root / uid
    dst = target_root / uid

    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)

print(f"Copied {len(matched_uids)} folders.")

Copied 1473 folders.


In [34]:
all_data_final = all_data_clean[
    all_data_clean['SeriesUID'].isin(matched_uids)
].copy()

print(all_data_final.shape)
print(all_data_final['pathology'].value_counts())

(1555, 15)
pathology
MALIGNANT    784
BENIGN       771
Name: count, dtype: int64


In [35]:
print(all_data_final['SeriesUID'].nunique())

1473


### Duplicate Rows Issue 

All Series UID have matached but still Rows in parent file are 1555 suggesting there are 82 duplicates. Let's explore them first and then decide how we will do it? 

In [36]:
print(len(matched_uids))
print(all_data_clean['SeriesUID'].nunique())

1473
1473


In [37]:
all_data_final.shape

(1555, 15)

In [38]:
uid_counts = all_data_clean['SeriesUID'].value_counts()

duplicate_uids = uid_counts[uid_counts > 1]

print("Duplicate UIDs:", len(duplicate_uids))
print(duplicate_uids.head(5))

Duplicate UIDs: 59
SeriesUID
1.3.6.1.4.1.9590.100.1.2.87251504411596839017815563663575708222     6
1.3.6.1.4.1.9590.100.1.2.354587724213018641829708719832963731890    5
1.3.6.1.4.1.9590.100.1.2.101999469712679926627011488331183444331    4
1.3.6.1.4.1.9590.100.1.2.292605978712963936606864280561587921668    4
1.3.6.1.4.1.9590.100.1.2.122014842013197980401803659680241641105    3
Name: count, dtype: int64


In [39]:
dup_rows = all_data_clean[
    all_data_clean['SeriesUID'].isin(duplicate_uids.index)
].sort_values('SeriesUID')

dup_rows.head(5)

,SeriesUID,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
26,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,1,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_1/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_1/1.3.6.1.4.1.9...
27,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,2,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_2/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_2/1.3.6.1.4.1.9...
28,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,3,mass,OVAL,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_3/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_3/1.3.6.1.4.1.9...
29,1.3.6.1.4.1.9590.100.1.2.101999469712679926627...,P_00044,2,RIGHT,CC,4,mass,LOBULATED,CIRCUMSCRIBED,3,BENIGN,5,Mass-Training_P_00044_RIGHT_CC/1.3.6.1.4.1.959...,Mass-Training_P_00044_RIGHT_CC_4/1.3.6.1.4.1.9...,Mass-Training_P_00044_RIGHT_CC_4/1.3.6.1.4.1.9...
310,1.3.6.1.4.1.9590.100.1.2.108388395511926748633...,P_00432,1,LEFT,MLO,2,mass,IRREGULAR,SPICULATED,4,MALIGNANT,3,Mass-Training_P_00432_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00432_LEFT_MLO_2/1.3.6.1.4.1.9...,Mass-Training_P_00432_LEFT_MLO_2/1.3.6.1.4.1.9...


The duplicates are for the same patient ID and are depending on the abnormalities. As our task is to classify Benign from Malignant, that's whu we will drop the duplicates keeping only one input to Target. 

In [40]:
all_data_image = all_data_clean.drop_duplicates(
    subset=['SeriesUID']
).copy()

In [41]:
all_data_image.shape

(1473, 15)